# Feature Engineering

I perform a chi-squared test to assess the statistical significance of the features and remove those that are not important, reducing noise in the dataset.

In [1]:
import pandas as pd
from scipy.stats import chi2_contingency
import os

In [2]:
data_path = os.path.join('..', 'data', '02_processed', 'churn_eda_processed.csv')
df = pd.read_csv(data_path)

In [3]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols.remove('Churn')
results = []

for col in cat_cols:
    table = pd.crosstab(df[col], df["Churn"])
    chi2, p, dof, expected = chi2_contingency(table)

    results.append({
        "Feature": col,
        "Chi2": chi2,
        "p_value": p
    })

results_df = pd.DataFrame(results).sort_values("p_value")
results_df

,Feature,Chi2,p_value
13,Contract,1179.545829,7.326182e-257
7,OnlineSecurity,846.677389,1.400687e-184
10,TechSupport,824.925564,7.407808e-180
6,InternetService,728.695614,5.831199e-159
15,PaymentMethod,645.429900,1.426310e-139
8,OnlineBackup,599.175185,7.776099e-131
9,DeviceProtection,555.880327,1.959389e-121
12,StreamingMovies,374.268432,5.353560e-82
11,StreamingTV,372.456502,1.324641e-81
14,PaperlessBilling,256.874908,8.236203e-58


I perform feature engineering to improve model performance and verify that all new features have been created and transformed successfully.

In [4]:
service_columns = [
    'MultipleLines',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'PhoneService'
]

#df.loc[:, 'Tenure_risk_cohorts'] = pd.cut(
#    df['tenure'], 
#    bins=[0, 4, 12, 36, 48, 100], 
#    labels=['risk_category', 'new_category', 'medium_category', 'established_category', 'loyal_category']
#)

df.loc[:, 'High_Risk_Tenure'] = pd.cut(
    df['tenure'], 
    bins=[0, 4, 12, 100], 
    labels=['high_risk_category', 'medium_risk_category', 'low_risk_category']
)

contract_stability = {'Month-to-month': 1, 'One year': 2, 'Two year': 3}
df['Contract_Stability'] = df['Contract'].map(contract_stability)

df['Fiber_No_Support'] = (
    (df['InternetService'] == 'Fiber optic') & 
    (df['TechSupport'] == 'No')
).astype(int)

df['Manual_Payment_Early'] = (
    (df['PaymentMethod'].isin(['Electronic check', 'Mailed check'])) & 
    (df['tenure'] <= 6)
).astype(int)

df['High_Risk_New_Montly'] = (
    (df['tenure'] <= 6) & 
    (df['Contract'] == 'Month-to-month')
).astype(int)

new_boolean_features = [
    'Fiber_No_Support',
    'Manual_Payment_Early',
    'High_Risk_New_Montly'
]

df[new_boolean_features] = df[new_boolean_features].astype(bool)

for i in df.columns:
    print(f"{i}: {df[i].unique()}")
print(df.dtypes)
df.isnull().sum()

gender: ['Female' 'Male']
SeniorCitizen: ['No' 'Yes']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
tenure: [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26 39]
PhoneService: ['No' 'Yes']
MultipleLines: ['No phone service' 'No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupport: ['No' 'Yes' 'No internet service']
StreamingTV: ['No' 'Yes' 'No internet service']
StreamingMovies: ['No' 'Yes' 'No internet service']
Contract: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling: ['Yes' 'No']
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
MonthlyCharges: [29.85 56.95 53.85 ... 63.1  44.2  78.7 ]
TotalC

gender                  0
SeniorCitizen           0
Partner                 0
Dependents              0
tenure                  0
PhoneService            0
MultipleLines           0
InternetService         0
OnlineSecurity          0
OnlineBackup            0
DeviceProtection        0
TechSupport             0
StreamingTV             0
StreamingMovies         0
Contract                0
PaperlessBilling        0
PaymentMethod           0
MonthlyCharges          0
TotalCharges            0
Churn                   0
High_Risk_Tenure        0
Contract_Stability      0
Fiber_No_Support        0
Manual_Payment_Early    0
High_Risk_New_Montly    0
dtype: int64

I drop columns identified as unnecessary by the chi-squared test, as well as features that are no longer needed following feature engineering.

In [5]:
df = df.drop(['gender', 'Contract'], axis=1)
df = df.drop(service_columns, axis=1)

In [6]:
pd_to_csv_path = os.path.join('..', 'data', '03_featured', 'churn_featured.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)

df.to_csv(pd_to_csv_path, index=False)

print(f"CSV saved to {pd_to_csv_path}")

CSV saved to ../data/03_featured/churn_featured.csv
